# JAX Core Patterns

Scientific Machine Learning (SciML) requires high-performance array computing, differentiable physics, and efficient solvers. In JAX, this relies on four core transformations:
1. **`jax.jit`**: Just-in-Time compilation for hardware acceleration.
2. **`jax.vmap`**: Auto-vectorization for batched evaluations.
3. **`jax.grad`**: Automatic differentiation for physics and optimization.
4. **`jax.lax.scan`**: Fast, compiled loops for time-stepping ODEs/PDEs.

In [1]:
import jax
import jax.numpy as jnp
import time

## Exercise 1: JIT Compilation for a Time-Step

In SciML, you often write a function to advance a system by one time step `dt`. If you don't JIT compile it, Python overhead will bottleneck your simulation.

**Task**: 
1. Implement a naive Euler step: $y_{n+1} = y_n + dt \cdot f(t, y_n)$ where $f(t, y) = -k \cdot y \cdot \exp(-y)$.
2. JIT compile the function.
3. Run both the un-jitted and jitted functions a few times and observe the speedup.

In [4]:
def euler_step(y, dt, k):
    # TODO: Implement the Euler step
    pass

y0 = jnp.array(1.0)
dt = 0.01
k = 1.5

# TODO: Time the un-jitted function

# TODO: Time the jitted function (remember to run it once first to compile, then time the second run!)

## Exercise 2: `vmap` for Parameter Sweeps & Initial Conditions

Instead of writing Python `for` loops to simulate a system over 10,000 different initial conditions or parameter variations, `vmap` pushes the loop down into optimized C++/CUDA code.

**Task**:
1. Create an array of 100,000 initial conditions for $y$.
2. Use `vmap` to apply your `jitted_euler_step` across all initial conditions simultaneously.
3. Ensure the parameter `k` and `dt` are not vectorized over (hint: use `in_axes`).

In [5]:
y0_array = jnp.linspace(0.1, 5.0, 100000)

# TODO: Create batched_step using jax.vmap

# TODO: Execute batched_step and verify the output shape is (100000,)

## Exercise 3: `grad`

In SciML, we often use models to approximate solutions to ODEs/PDEs. We need the gradient of the model *with respect to its inputs* (spatial/temporal coordinates) to approximate the physics, and the gradient *with respect to its weights* to update the model.

**Task**:
1. We have a dummy neural net prediction: `pred(w, t) = w[0]*t + w[1]*t**2`.
2. The governing ODE is $\frac{dy}{dt} = -y$.
3. Write a `physics_loss` function that computes the residual: $(\frac{dy}{dt} + y)^2$.
4. Compute the gradient of the loss with respect to the weights `w`.

In [6]:
def net(w, t):
    return w[0] * t + w[1] * t**2

def physics_loss(w, t):
    # TODO: Compute dy/dt using jax.grad, evaluate dy_dt_fn at (w, t), calculate and return the squared residual: (dy/dt + y)^2
    pass

weights = jnp.array([1.0, -0.5])
t_collocation = 0.5

## Exercise 4: `lax.scan` for Fast Time-Stepping
Standard Python `for` loops get unrolled during JIT compilation. If you simulate an ODE for 100,000 steps, compilation will take forever or crash. JAX provides `jax.lax.scan` to compile loops efficiently.

**Task**:
1. Write an ODE solver using `jax.lax.scan` that rolls out `euler_step` for `num_steps`.
2. `scan` requires a step function with the signature: `carry, y = step_fn(carry, x)`. Here, the `carry` is our state $y$, and `x` could be the time steps (or `None` if we don't need them).

In [ ]:
def solve_ode(y0, dt, k, num_steps):
    
    # TODO: Define the body of the scan loop
    def scan_step(y_current, _):
        # 1. Compute the next y using euler_step
        # 2. Return (y_next, y_next) -> (carry_over, saved_output)
        pass
    pass

# TODO: JIT compile the solve_ode function
# TODO: Run fast_solver for 1000 steps and print the final value.

## Exercise 5: PRNG

We are going to estimate $\pi$ by throwing darts at a board. We want to do this as fast as possible, but we can't just throw $M*N$ darts in one massive array (which might OOM your GPU). 

Instead, we will define a function that throws $N$ darts, and we will `vmap` that function over $M$ independent PRNG keys to run $M$ simulations completely in parallel.

**Task**:
1. Write the logic for `single_block_mc` to estimate $\pi$.
2. Combine `jax.jit` and `jax.vmap` to vectorize the simulation over an array of keys.
3. Time the compiled execution.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as random
import time

def single_block_mc(key, num_samples):
    """Runs a single Monte Carlo block to estimate pi."""
    # get two independent coordinates for x and y between -1.0 and 1.0

    x = random.uniform(key_x, shape=(num_samples,), minval=-1.0, maxval=1.0)
    y = random.uniform(key_y, shape=(num_samples,), minval=-1.0, maxval=1.0)
    
    # Check which points fall inside the unit circle (x^2 + y^2 <= 1.0)
    
    # Calculate the estimate for pi: 4 * (inside / total)
    
    # return pi_estimate
    pass

In [ ]:
M = 10_000  # Number of parallel block
N = 10_000  # Number of samples per block

master_key = random.key(42)